# Coupon A/B Test Analysis

## 1. Data ingestion

We download and extract the raw datasets (orders, consumers, restaurants and A/B test reference) using the reusable functions in `src/ingestion.py`. The orders file is large (~3.6M rows), so we rely on streaming / chunked loading instead of reading everything into memory at once.


## 2. Data cleaning

Using `src/processing.py`, we standardize schemas, convert timestamps, remove duplicate orders, filter out invalid values (e.g., non-positive order amounts) and handle missing values. The final output of this step is the `fact_orders` table with the required columns and the treatment flag `is_target` merged from the A/B reference.


## 3. Feature engineering

From `fact_orders` we compute customer-level RFM-style metrics in `src/feature_engineering.py`. These include total orders, total spent, average order value, first/last order dates, recency (days since last order), frequency and monetary value. These features feed both the A/B analysis (per user) and the segmentation step.


## 4. A/B test analysis

We compute A/B metrics in `src/ab_test.py`: conversion rate, orders per user, GMV, GMV per user and average order value, comparing target vs control. We apply a Welch t-test on order values and a proportion z-test on conversion rates, reporting p-values and confidence intervals to assess statistical significance.


## 5. Segmentation

Using `src/segmentation.py`, we build RFM-based segments (Champions, Loyal, Potential, At Risk, Low Engagement). We then analyze A/B test performance by segment to understand which user profiles respond best to the coupon campaign, and compute uplift in GMV per user for each segment.


## 6. Business insights

In the final section we interpret the metrics and statistical tests to answer key business questions:

- Did the coupon campaign significantly improve conversion and/or revenue?
- Is the campaign financially viable after accounting for coupon costs and iFood's take rate?
- Which segments show the highest uplift and should be prioritized for future campaigns?

The corresponding tables are saved in `outputs/tables/` and charts (conversion, GMV per user, order value distributions, uplift by segment and RFM distributions) are saved in `outputs/charts/`.
